# GameTheory-16c : La dimension paiement que le designer n'a jamais cherchée — extraction de revenu sous contraintes locales au premier meilleur

**Objectifs** : énumérer les paiements, comparer les frontières et expliciter les contraintes du jouet.

**Prérequis** : mécanismes binaires, utilité quasi-linéaire ; **durée** : 75 minutes.

**Navigation** : [Version Python](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) | [<< 16b-Automated-Mechanism-Design](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) | [Index GameTheory](README.md) | [16k-Encheres-Myerson >>](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb) | [jumeau Rust](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Rust.ipynb)

GT-16b construit un générateur de tables, un vérificateur indépendant et un témoin d'impossibilité. Le générateur ne cherche que 16 tables d'issues et fixe les paiements à zéro. Ici l'énumération couvre les 4096 mécanismes : sur l'instance (1,1), 47 satisfont les contraintes locales, dont 34 au bien-être maximal et 29 avec un revenu positif. Le générateur atteint bien J=2 et un revenu nul, mais sa table échoue au test local de non-déviation ; la sortie montre la déviation profitable.

Deux variantes de participation sont comparées : l'IR à types fixés, appelée point-prior dans ce jouet, et l'IR uniforme sur tous les types et tous les reports. La seconde force ici les paiements à zéro, alors que la première permet l'extraction complète sur (1,1).

**Portée des vérificateurs.** La non-déviation locale fixe les deux types et compare chaque changement unilatéral depuis chaque profil de reports. Comme les deux sens d'une même arête sont contrôlés, elle impose l'égalité des utilités entre reports propres. Le DSIC usuel compare, pour chaque type et chaque report des autres, le report sincère à ses déviations ; ce n'est pas le test implémenté ici. L'IR uniforme ne calcule aucune espérance et n'est pas l'IR ex-interim usuelle.

La référence pédagogique primaire est le source Python versionné cité en fin de carnet. Les publications historiques qui y figuraient sont conservées comme contexte, sans attribuer les résultats de ce jouet à leurs théorèmes.

## Objectifs d'apprentissage

1. Lire les 16 candidats du générateur et son choix de paiements nuls.
2. Énumérer les 4096 mécanismes et mesurer la frontière revenu/bien-être sur les quatre instances à types fixés.
3. Faire vérifier le générateur et le mécanisme extracteur par le même organe indépendant.
4. Distinguer l'IR à types fixés de l'IR uniforme sur types et reports.
5. Borner les conclusions au modèle exécuté, sans confondre sa non-déviation locale avec le DSIC usuel.

## 1. Ce que le designer committé designe vraiment

Le domaine de GT-16b : 2 agents, types θ ∈ {0,1}, issue o ∈ {0,1}, paiements ∈ {0,1}². Un mécanisme est un couple de tables : `issue(r) ∈ {0,1}` et `paiement(r) ∈ {0,1}²` pour chacun des 4 profils de reports `r`. L'espace COMPLET des mécanismes compte `2⁴ × 4⁴ = 4096` candidats. Le générateur de GT-16b n'en visite que `2⁴ = 16` : il énumère les issues, prend l'argmax de bien-être, puis **fixe tous les paiements à zéro** (« les paiements minimaux qui satisfont trivialement IR », dit son commentaire). Le choix est défendable — mais c'est un choix : l'utilité quasi-linéaire `uᵢ = θᵢ·issue − paiementᵢ` rend les transferts neutres pour le bien-être `J = Σθᵢ·issue`, donc ignorer les paiements ne coûte rien au critère optimisé… et coûte tout au seul autre critère naturel, le revenu.

Reproduisons d'abord le designer committé sur l'instance `true = (1,1)` — le point-prior où les deux agents sont de type 1.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
CultureInfo.DefaultThreadCurrentCulture=CultureInfo.InvariantCulture;
Console.WriteLine($"Runtime : {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription}");
record Mechanisme(int[] Issues,int[,] Paiements);
int Type(int profil,int agent)=>(profil>>(1-agent))&1;
int[] Issues(int masque)=>Enumerable.Range(0,4).Select(r=>(masque>>(3-r))&1).ToArray();
IEnumerable<Mechanisme> EnumererTous(int maximum=1,int minimum=0){
 int radix=maximum-minimum+1; int count=(int)Math.Pow(radix,8);
 for(int mask=0;mask<16;mask++)for(int number=0;number<count;number++){var pay=new int[4,2];int q=number;for(int k=7;k>=0;k--){pay[k/2,k%2]=minimum+q%radix;q/=radix;}yield return new Mechanisme(Issues(mask),pay);}
}
(Mechanisme mecanisme,int bienEtre) GenererGT16b(int type0,int type1){int best=-1,profil=2*type0+type1;int[] choix=null;for(int mask=0;mask<16;mask++){int o=Issues(mask)[profil],j=(type0+type1)*o;if(j>best){best=j;choix=Issues(mask);}}return(new Mechanisme(choix,new int[4,2]),best);}
var generateur=GenererGT16b(1,1);
Console.WriteLine($"Instance point-prior : true=(1,1), issue={generateur.mecanisme.Issues[3]}, bien-être={generateur.bienEtre}, revenu=0");
Console.WriteLine("Générateur : 16 tables d'issues ; espace complet : 16 × 256 = 4096 mécanismes");

Runtime : .NET 10.0.14


Instance point-prior : true=(1,1), issue=1, bien-être=2, revenu=0


Générateur : 16 tables d'issues ; espace complet : 16 × 256 = 4096 mécanismes


**Lecture de la sortie.** Le générateur atteint le bien-être maximal J=2 sur (1,1), avec un revenu nul. Il visite 16 tables d'issues sur 4096 mécanismes, soit environ 0,4% de l'espace. Cela ne prouve pas l'admissibilité : la section suivante soumet réellement sa table au vérificateur local et constate son échec. L'énumération conjointe sépare ainsi trois questions : bien-être, revenu et respect des contraintes.

## 2. L'énumération conjointe — le vérificateur hérité tranche

On hérite l'organe de GT-16b : son **vérificateur indépendant**, en copie locale, zéro import du générateur — mêmes définitions de l'utilité quasi-linéaire, de non-déviation locale (aucune déviation unilatérale strictement profitable, pour chaque agent et chaque profil reporté) et d'IR (utilité ≥ 0 pour chaque agent et chaque profil, **aux types vrais** — la sémantique exacte de `verify_IR` de GT-16b). Puis on énumère les 4096 candidats de l'espace complet, et on compte par instance point-prior : les admissibles (non-déviation locale ∧ IR), parmi eux les à bien-être optimal, parmi ceux-ci les à revenu strictement positif, et le revenu maximal.

In [2]:
bool VerifierDsicLocal(Mechanisme m,int type0,int type1){for(int r=0;r<4;r++)for(int i=0;i<2;i++){int theta=i==0?type0:type1;int autre=r^(1<<(1-i));int sincere=theta*m.Issues[r]-m.Paiements[r,i],dev=theta*m.Issues[autre]-m.Paiements[autre,i];if(dev>sincere)return false;}return true;}
bool VerifierIR(Mechanisme m,int type0,int type1){for(int r=0;r<4;r++)for(int i=0;i<2;i++)if((i==0?type0:type1)*m.Issues[r]-m.Paiements[r,i]<0)return false;return true;}
int BienEtre(Mechanisme m,int type0,int type1)=>(type0+type1)*m.Issues[2*type0+type1];
int Revenu(Mechanisme m,int type0,int type1)=>m.Paiements[2*type0+type1,0]+m.Paiements[2*type0+type1,1];
var frontiere=new Dictionary<int,(int admissibles,int premierMeilleur,int revenuPositif,int revenuMax)>();
Console.WriteLine("Types | admissibles | premier meilleur | revenu positif | revenu maximal");
for(int profil=0;profil<4;profil++){int t0=Type(profil,0),t1=Type(profil,1);int admissibles=0,fb=0,pos=0,best=-1;foreach(var m in EnumererTous()){if(VerifierDsicLocal(m,t0,t1)&&VerifierIR(m,t0,t1)){admissibles++;if(BienEtre(m,t0,t1)==t0+t1){fb++;int rev=Revenu(m,t0,t1);if(rev>0)pos++;best=Math.Max(best,rev);}}}frontiere[profil]=(admissibles,fb,pos,best);Console.WriteLine($"({t0},{t1}) | {admissibles} | {fb} | {pos} | {best}");}
Console.WriteLine("Vérificateur local : aucune déviation profitable aux types fixés, IR sur tous les reports");
Console.WriteLine($"Générateur hérité : non-déviation locale={VerifierDsicLocal(generateur.mecanisme,1,1)}, IR={VerifierIR(generateur.mecanisme,1,1)} ; agent 0 : report (0,1), utilité=0, déviation (1,1), utilité=1");

Types | admissibles | premier meilleur | revenu positif | revenu maximal


(0,0) | 16 | 16 | 0 | 0


(0,1) | 25 | 15 | 10 | 1


(1,0) | 25 | 15 | 10 | 1


(1,1) | 47 | 34 | 29 | 2


Vérificateur local : aucune déviation profitable aux types fixés, IR sur tous les reports


Générateur hérité : non-déviation locale=False, IR=True ; agent 0 : report (0,1), utilité=0, déviation (1,1), utilité=1


**Lecture de la sortie.** Dans l'ordre(0,0),(0,1),(1,0),(1,1), les nombres de mécanismes admissibles sont 16, 25, 25, 47. Sur(1,1), 34 atteignent le bien-être maximal ; 29 de ceux-ci ont un revenu positif et le maximum vaut 2. Le générateur hérité atteint lui aussi J=2, mais sa table viole la non-déviation locale : à types (1,1), l'agent 0 préfère le report (1,1) au report (0,1), pour une utilité 1 au lieu de 0. Son revenu nul et son défaut d'admissibilité sont deux constats distincts.

Sur(0,0), l'IR et les paiements non négatifs imposent des paiements nuls : les 16 tables d'issues sont admissibles et aucun revenu positif n'existe. Le revenu maximal au premier meilleur vaut respectivement0,1,1,2 sur les quatre instances, soit la somme des types dans ce modèle.

## Exercice 1 : l'instance (0,0) est une forteresse

La table de la section 2 montre 16 mécanismes admissibles sur `(0,0)` et aucun à revenu positif. Démontrez-le proprement : pour chaque agent de type 0, écrivez la contrainte IR point-prior et montrez qu'elle force `paiementᵢ(r) = 0` pour tout profil `r` — indépendamment de l'issue. Concluez : le revenu maximal d'une instance est nul dès que tous les types sont nuls, et la forme close `Σθᵢ` est cohérente. Vérifiez enfin votre preuve par énumération sur les deux autres instances à agent de type 0 : leur revenu ne peut venir que de l'agent intéressé.

Complétez `borne_revenu` (Python) ou `borneRevenu` (C#), pour deux types binaires et une issue binaire ; retournez la borne totale de revenu impliquée par IR. Le garde mélange cas nuls et non nuls, puis votre preuve explique pourquoi un type nul ne finance rien.

In [3]:
// Exercice 1 a completer
// 1. Pour true = (0,0) : ecrire la contrainte IR d'un agent de type 0 (u_i = 0*issue - p_i)
// 2. En deduire p_i(r) = 0 pour tout r, puis revenu = 0 pour TOUT mecanisme admissible
// 3. Verifier : FRONTIERE[(0,0)][2] == 0 et FRONTIERE[(0,0)][3] == 0 ; puis (0,1) : revenu <= 1
// Indice : u_i = -p_i(r) >= 0 et p_i(r) dans {0,1} => p_i(r) = 0
Func<int,int,int,double?> borneRevenu=(type0,type1,issue)=>{
 // TODO : borne issue de l'IR pour les types fixés et l'issue donnée
 return null;
};
var casBorne=new[]{(0,0,0,0.0),(0,0,1,0.0),(0,1,1,1.0),(1,0,1,1.0),(1,1,1,2.0),(1,1,0,0.0)};
var reponsesBorne=casBorne.Select(c=>borneRevenu(c.Item1,c.Item2,c.Item3)).ToArray();
if(reponsesBorne.All(x=>x is null))Console.WriteLine("Exercice 1 à compléter : borne IR et preuve pour les types nuls");
else {for(int k=0;k<casBorne.Length;k++)if(reponsesBorne[k] is null||!double.IsFinite(reponsesBorne[k].Value)||Math.Abs(reponsesBorne[k].Value-casBorne[k].Item4)>1e-9)throw new Exception("Borne incorrecte");Console.WriteLine("Exercice 1 : bornes validées ; rédigez la preuve et relisez la frontière (0,0), (0,1), (1,0)");}

Exercice 1 à compléter : borne IR et preuve pour les types nuls


## 3. Le mécanisme extrateur — exhibé et vérifié par l'organe hérité

Sur l'instance (1,1), l'extracteur produit l'issue 1 et facture 1 à chaque agent au report (1,1) ; les autres profils donnent l'issue 0 et des paiements nuls. Pour les types fixés à 1, chaque agent a une utilité nulle à tous les reports. La non-déviation locale est donc satisfaite par égalité, l'IR est serrée, le bien-être vaut 2 et le revenu vaut 2. La cellule suivante fait réellement ces vérifications.

Cette propriété concerne les types (1,1). Elle ne donne aucune garantie d'incitation ou de participation pour d'autres types sous la définition habituelle de DSIC.

In [4]:
var extracteur=new Mechanisme(new[]{0,0,0,1},new int[,]{{0,0},{0,0},{0,0},{1,1}});
Console.WriteLine($"Extracteur : non-déviation locale={VerifierDsicLocal(extracteur,1,1)}, IR={VerifierIR(extracteur,1,1)}, bien-être={BienEtre(extracteur,1,1)}, revenu={Revenu(extracteur,1,1)}");
int sincere=extracteur.Issues[3]-extracteur.Paiements[3,0],deviation=extracteur.Issues[1]-extracteur.Paiements[1,0];
Console.WriteLine($"Agent 1 : utilité sincère={sincere}, après déviation={deviation} ; indifférence faible");
Console.WriteLine($"Contre-témoin : même bien-être {generateur.bienEtre}, revenu {Revenu(extracteur,1,1)} contre 0 pour le générateur");

Extracteur : non-déviation locale=True, IR=True, bien-être=2, revenu=2


Agent 1 : utilité sincère=0, après déviation=0 ; indifférence faible


Contre-témoin : même bien-être 2, revenu 2 contre 0 pour le générateur


**Lecture de la sortie.** L'extracteur passe la non-déviation locale et l'IR sur les types (1,1), avec bien-être 2 et revenu 2. L'exemple de changement de report donne une utilité 0 de part et d'autre ; le vérificateur contrôle également les autres profils. Le designer connaît les types pour cette expérience : il s'agit d'information complète de son point de vue. Aucun résultat d'extraction à prior distribué n'est démontré ou reproduit.

## Exercice 2 : élargir la gamme des paiements

Toute la -c utilise des paiements ∈ {0, 1}. Sur l'instance `(1,1)`, élargissez à {0, 1, 2} et relancez l'énumération conjointe (l'espace croît à `2⁴ × 9⁴` — reste trivial). Écrivez d'abord la borne : IR point-prior pour un agent de type 1 exige `paiementᵢ(r) ≤ issue(r) ≤ 1` — le revenu au profil vrai est donc plafonné à 2, quelle que soit la gamme. Confirmez par l'énumération, puis observez ce que la gamme étendue ne change PAS : le bien-être `J` ne compte pas les paiements — dites ce que ça révèle de la définition du bien-être du jouet (un transfert de l'agent au designer y est invisible : ni coût, ni benefice social).

Complétez `mesurer_frontiere` / `mesurerFrontiere` : consommez l’itérable des mécanismes fourni et retournez le quadruplet (admissibles, premier meilleur, revenu positif, revenu maximal). Le garde observe le nombre de candidats réellement parcourus, dont les 104 976 candidats de la gamme étendue.

In [5]:
// Exercice 2 a completer
// 1. Prediction ecrite AVANT : max revenu @ fb sur (1,1) avec paiements dans {0,1,2} = ?
// 2. Enumerer (issues 2^4) x (paiements {0,1,2}^(2*4)) avec le meme verificateur
// 3. Comparer au max = 2 : la borne vient de IR (p_i <= issue <= 1), pas de la gamme
// 4. Bonus : J change-t-il avec les paiements ? (J ne compte que les utilites grossieres)
// Indice : pour l'agent i de type 1, IR en r exige 1*issue(r) - p_i(r) >= 0, et issue(r) <= 1
Func<IEnumerable<Mechanisme>,int,int,(int,int,int,int)?> mesurerFrontiere=(candidats,type0,type1)=>{
 // TODO : consommer l'énumération et compter admissibles, premier meilleur, revenu positif et revenu maximal
 return null;
};
var casFrontiere=new[]{(1,0,0),(1,0,1),(1,1,0),(1,1,1),(2,1,1),(2,0,1)};
int completes=0;
foreach(var (maxPaiement,t0,t1) in casFrontiere){int visites=0;IEnumerable<Mechanisme> Observer(){foreach(var m in EnumererTous(maxPaiement)){visites++;yield return m;}}var rep=mesurerFrontiere(Observer(),t0,t1);if(rep is null)continue;completes++;var f=frontiere[2*t0+t1];if(rep.Value!=(f.admissibles,f.premierMeilleur,f.revenuPositif,f.revenuMax)||visites!=16*(int)Math.Pow(maxPaiement+1,8))throw new Exception("Frontière ou budget incorrect");}
if(completes==0)Console.WriteLine("Exercice 2 à compléter : gamme de paiements étendue");else if(completes!=casFrontiere.Length)throw new Exception("Réponse partielle");else Console.WriteLine("Exercice 2 : six frontières et budgets d'énumération validés");

Exercice 2 à compléter : gamme de paiements étendue


## 4. La couture du témoin d'impossibilité — deux lectures d'IR qui ne se rencontrent pas

Le témoin hérité interdit les paiements au moins égaux à 1 à tous les reports lorsqu'un agent de type 0 doit participer : son utilité serait alors au plus −1. L'IR à types fixés applique cette contrainte seulement aux types présents. Sur(1,1), aucun type nul n'est présent ; l'énumération doit donc vérifier si un paiement uniforme survit.

- **IR à types fixés (point-prior dans le jouet)** : pour chaque profil de reports et chaque agent, l'utilité de son type fixé doit être positive ou nulle.
- **IR uniforme sur types et reports** : pour chaque profil, agent et type possible, l'utilité doit être positive ou nulle. C'est une contrainte robuste, y compris quand type et report diffèrent.

La seconde n'est pas l'IR ex-interim usuelle : aucune espérance conditionnelle sur les types des autres agents n'est calculée. On conserve les deux expériences et on mesure leurs conséquences séparément.

In [6]:
Console.WriteLine("Paiements ≥1 partout : IR point-prior, instance par instance");
for(int profil=0;profil<4;profil++){int t0=Type(profil,0),t1=Type(profil,1);int count=EnumererTous(2,1).Count(m=>VerifierDsicLocal(m,t0,t1)&&VerifierIR(m,t0,t1));Console.WriteLine($"({t0},{t1}) : {count} admissible(s)");}
var survivant=new Mechanisme(Enumerable.Repeat(1,4).ToArray(),new int[,]{{1,1},{1,1},{1,1},{1,1}});
Console.WriteLine($"Survivant uniforme (1,1) : non-déviation locale={VerifierDsicLocal(survivant,1,1)}, IR={VerifierIR(survivant,1,1)}, bien-être={BienEtre(survivant,1,1)}, revenu={Revenu(survivant,1,1)}");
bool VerifierIRUniforme(Mechanisme m){for(int r=0;r<4;r++)for(int i=0;i<2;i++)for(int theta=0;theta<2;theta++)if(theta*m.Issues[r]-m.Paiements[r,i]<0)return false;return true;}
var robustes=EnumererTous().Where(m=>VerifierIRUniforme(m)&&VerifierDsicLocal(m,1,1)).ToArray();
int nonNuls=robustes.Count(m=>Enumerable.Range(0,8).Any(k=>m.Paiements[k/2,k%2]!=0));
Console.WriteLine($"IR uniforme sur types et reports : {robustes.Length} admissibles, {nonNuls} avec paiement non nul ; revenus=[{string.Join(",",robustes.Select(m=>Revenu(m,1,1)).Distinct().Order())}]");
Console.WriteLine("La contrainte robuste inclut theta=0 à chaque report : tout paiement est nul. Ce n'est pas l'IR ex-interim usuelle.");

Paiements ≥1 partout : IR point-prior, instance par instance


(0,0) : 0 admissible(s)


(0,1) : 0 admissible(s)


(1,0) : 0 admissible(s)


(1,1) : 1 admissible(s)


Survivant uniforme (1,1) : non-déviation locale=True, IR=True, bien-être=2, revenu=2


IR uniforme sur types et reports : 2 admissibles, 0 avec paiement non nul ; revenus=[0]


La contrainte robuste inclut theta=0 à chaque report : tout paiement est nul. Ce n'est pas l'IR ex-interim usuelle.


**Lecture de la sortie.** La contrainte de paiement positif partout ne laisse aucun mécanisme admissible sur les trois instances contenant un type 0. Sur(1,1), le mécanisme uniforme issue 1/paiements(1,1) survit et reçoit un revenu 2. La conclusion d'ensemble vide doit donc être bornée à la contrainte de participation réellement employée.

Avec l'IR uniforme sur tous les types et reports, le type 0 est testé à chaque report : les paiements non négatifs sont alors tous nuls. Ce résultat provient du quantificateur supplémentaire, pas d'un calcul d'espérance ni d'un théorème extérieur au modèle.

## Exercice 3 : non-déviation locale stricte

Construisez les marges signées du vérificateur local : utilité au report courant moins utilité après changement unilatéral. Une marge positive favorise le report courant, une marge négative rend le changement profitable. Retournez les huit marges dans `marges_locales` / `margesLocales`, en parcourant les quatre profils puis les deux agents. Les utilités de ce domaine sont entières : les marges attendues le sont aussi.

Les trois fixtures contiennent des marges négatives, nulles et positives. Le garde vérifie ensuite chaque mécanisme admissible au bien-être maximal et revenu positif. Démontrez pourquoi le contrôle des deux sens d'une même arête empêche toutes les marges d'être strictement positives ; reliez cette preuve au comptage obtenu lorsque votre fonction est complétée. Le cas de l'extracteur et celui du mécanisme uniforme doivent être interprétés dans le modèle local, sans les confondre avec un test DSIC standard.

In [7]:
// Exercice 3 a completer
// 1. Definir verif_dsic_strict(M, true) : u_sincere > u_dev pour toute deviation
// 2. Re-enumerer l'espace complet sur (1,1) : compter les non-déviation locale-strict + IR a welfare optimal
// 3. Max revenu sous non-déviation locale strict vs 2 (faible) ; et le survivant uniforme de la section 4 ?
// Indice : sur ce domaine, les ecarts d'utilite sont entiers -- l'egalite exacte est le seul cas filtre
Func<Mechanisme,int,int,double[]?> margesLocales=(m,type0,type1)=>{
 // TODO : huit marges utilité courante moins utilité après changement de report ; ordre profil puis agent
 return null;
};
var sansPaiement=new Mechanisme(new[]{0,0,0,1},new int[4,2]);
var casMarges=new[]{(extracteur,new double[8]),(survivant,new double[8]),(sansPaiement,new[]{0.0,0.0,-1.0,0.0,0.0,-1.0,1.0,1.0})};
int reponsesMarges=0;foreach(var (m,attendu) in casMarges){var rep=margesLocales(m,1,1);if(rep is null)continue;reponsesMarges++;if(rep.Length!=8||rep.Any(x=>!double.IsFinite(x)||x!=Math.Round(x))||rep.Where((x,k)=>Math.Abs(x-attendu[k])>1e-9).Any())throw new Exception("Marges incorrectes");}
if(reponsesMarges==0)Console.WriteLine("Exercice 3 à compléter : non-déviation stricte locale");else if(reponsesMarges!=casMarges.Length)throw new Exception("Réponse partielle");else {int stricts=EnumererTous().Where(m=>VerifierDsicLocal(m,1,1)&&VerifierIR(m,1,1)&&BienEtre(m,1,1)==2&&Revenu(m,1,1)>0).Count(m=>{var ms=margesLocales(m,1,1);if(ms is null||ms.Length!=8||ms.Any(x=>!double.IsFinite(x)))throw new Exception("Marges incomplètes");if(ms.Any(x=>x!=0.0))throw new Exception("Marges incompatibles avec le vérificateur local : les deux sens imposent une égalité");return ms.All(x=>x>0);});Console.WriteLine($"Exercice 3 : marges validées, {stricts} mécanismes à revenu positif satisfont toutes les inégalités strictes locales");}

Exercice 3 à compléter : non-déviation stricte locale



(6,33): warning CS8632: The annotation for nullable reference types should only be used in code within a '#nullable' annotations context.



## 5. Verdict — ce que l'effondrement coûte, ce qui survit

| Question | Réponse mesurée | Portée |
|---|---|---|
| Le générateur est-il admissible ? | Non : l'IR passe, mais la non-déviation locale échoue | Types fixés(1,1), table complète vérifiée |
| Son bien-être et son revenu ? | Bien-être2, revenu 0 | Il maximise seulement le premier critère |
| Que permet l'espace complet ? |47 admissibles, 34 au premier meilleur,29 à revenu positif, maximum2 | Instance(1,1), paiements binaires |
| Le paiement uniforme positif est-il impossible ? | Oui sur les trois instances avec un type 0 ; un survivant sur (1,1) | IR à types fixés |
| Que change l'IR uniforme ? | Tout paiement non nul est exclu | Tous types et tous reports sont testés |
| Revenu maximal au premier meilleur ? |0,1,1,2 dans l'ordre des quatre instances | Égal à la somme des types dans ce jouet |

Le même vérificateur révèle le défaut du générateur et certifie les mécanismes issus de l'énumération. Le critère optimisé et les contraintes doivent donc être contrôlés séparément, sur le livrable réel.

### Questions ouvertes — ce que ce jouet ne dit pas

- **Types distribués.** Comment changer le problème quand le designer ne connaît plus les types présents ? Il faut définir un prior, la participation et les incitations correspondantes avant de comparer les revenus ; cette extension n'est pas exécutée ici.
- **Coût de l'issue.** La provision de l'issue 1 est gratuite dans ce modèle. Ajouter un coût c exige de distinguer revenu, coût et surplus avant de chercher un mécanisme qui finance la provision.
- **Autre définition des incitations.** Comment remplacer la comparaison de tous les reports à types fixés par une comparaison entre report sincère et déviations pour chaque type ? Le résultat local d'impossibilité de la stricte positivité des deux sens ne doit pas être transféré sans refaire cette définition.

## Exercices — récapitulatif

Les trois activités réparties démontrent la borne IR des types nuls, mesurent la gamme de paiements étendue en conservant son budget, puis construisent les marges de non-déviation stricte. Rédigez les arguments qui relient vos sorties aux limites du modèle.

## À retenir

1. Le générateur visite 16 tables d'issues sur 4096 mécanismes. Il atteint le bien-être 2 et un revenu nul sur (1,1), mais sa table échoue au test local de non-déviation.
2. L'énumération complète trouve 47 admissibles, 34 au bien-être maximal et 29 à revenu positif sur (1,1). Le revenu maximal y vaut 2, atteint par l'extracteur ; sur les quatre instances il vaut 0,1,1,2.
3. L'extracteur a des utilités locales nulles pour les types fixés(1,1). Cela ne démontre pas une propriété générale de DSIC. La stricte positivité des marges dans les deux sens d'une arête est impossible.
4. L'IR à types fixés et l'IR uniforme sur tous les types/reports n'ont pas le même ensemble admissible. La seconde impose des paiements nuls ; ce n'est pas une IR calculée en espérance.

La méthode est de mesurer tout l'espace annoncé, d'employer un vérificateur indépendant et de nommer exactement les quantificateurs des contraintes. Les exercices gardent leurs fonctions à compléter pour vérifier les bornes, les budgets de l'énumération étendue et les marges signées.

## Vérifiez votre compréhension

1. Le générateur atteint J=2. Pourquoi cela ne suffit-il pas ?

<details><summary>Réponse</summary>Il optimise seulement le bien-être. Sa table et ses paiements nuls donnent bien J=2, mais le vérificateur local affiche une déviation profitable : pour les types (1,1), l'agent 0 gagne 1 en passant du report (0,1) au report (1,1). L'IR passe et le revenu reste nul ; ces trois tests répondent à des questions différentes.</details>

2. Que signifie l'indifférence de l'extracteur ?

<details><summary>Réponse</summary>Dans l'expérience aux types fixés(1,1), toute utilité locale vaut 0, aux quatre reports : les inégalités des deux sens sont donc des égalités. Le test ne prouve pas une incitation générale au report sincère ni une robustesse hors de ce modèle. Par exemple, si un agent était de type 0 et l'autre reportait1, passer de son report 0 à 1 le ferait payer 1 pour une valeur nulle : il perdrait 1, et non 0.</details>

3. Pourquoi le paiement uniforme survit-il sur (1,1) mais pas sous IR uniforme ?

<details><summary>Réponse</summary>À types fixés(1,1), issue 1 et paiement 1 donnent une utilité nulle à chacun. L'IR uniforme teste aussi le type 0 au même report ; son utilité serait−1, donc le mécanisme est rejeté. C'est un changement de quantificateur, sans moyenne sur une distribution.</details>

4. Le revenu maximal égal à la somme des types est-il un résultat général d'AMD ?

<details><summary>Réponse</summary>Non. Il est atteint ici dans un domaine binaire, aux types connus du designer et avec les contraintes locales exécutées. D'autres priors, coûts ou définitions des incitations constituent d'autres problèmes à définir et à résoudre.</details>

## Pour aller plus loin

**Référence pédagogique primaire.** Le calcul porté et relu est le [source Python versionné à a8566836](https://github.com/jsboige/CoursIA/blob/a8566836bb4246cfedbf6923f63fb0b2ab37d368/MyIA.AI.Notebooks/GameTheory/GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb), avec ses énumérations et ses vérificateurs explicites. Les expériences du présent carnet vérifient ce modèle local, pas les théorèmes d’une publication externe.

**Contexte historique conservé du source.** Les notices ci-dessous étaient présentes dans ce source et situent le thème ; aucune nouvelle lecture de copie archivée ni vérification Crossref de première main n’est revendiquée par ce port. L’IR robuste de ce jouet ne reproduit pas l’IR ex-interim usuelle.

- Green, J., & Laffont, J.-J. (1977). Characterization of satisfactory mechanisms for the revelation of preferences for public goods. *Econometrica*, 45(2), 427. https://doi.org/10.2307/1911219
- Rochet, J.-C. (1987). A necessary and sufficient condition for rationalizability in a quasi-linear context. *Journal of Mathematical Economics*, 16(2), 191-200. https://doi.org/10.1016/0304-4068(87)90007-3
- Saijo, T. (1988). Strategy space reduction in Maskin's theorem: sufficient conditions for Nash implementation. *Econometrica*, 56(3), 693. https://doi.org/10.2307/1911706
- Myerson, R. B., & Satterthwaite, M. A. (1983). Efficient mechanisms for bilateral trading. *Journal of Economic Theory*, 29(2), 265-281. https://doi.org/10.1016/0022-0531(83)90048-0

**Carnet suivant.** [GameTheory-16d — Joueurs LLM (Othman-Sandholm)](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) -- du designer qui optimise dans un espace borné aux **joueurs** qui ne sont pas scriptés : le gel pré-enregistré et les contrôles scriptés y jouent le rôle que le vérificateur séparé joue ici.

**Carnets voisins.**

- [GameTheory-16b — Automated Mechanism Design](GameTheory-16b-Automated-Mechanism-Design-Python.ipynb) -- l'organe repris tel quel, et le générateur dont le carnet mesure l'effondrement.
- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : principe de révélation, VCG, et les limites que le théorème d'impossibilité pose à toute conception de mécanisme.
- [GameTheory-16k — Théorie des enchères (Myerson)](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb) -- le même Myerson, côté vendeur : la réserve qui retire la rente d'information.
